# Experimento 4C: PubMedBERT + class weights

**Objetivo:** aislar el efecto de cambiar `CrossEntropyLoss()` por
`CrossEntropyLoss(weight=...)` con pesos de frecuencia inversa (formula
"balanced" de `sklearn.utils.class_weight`: `weight[i] = n_total / (n_clases *
count[i])`), manteniendo todo lo demas igual -- mismo neg_ratio=3, mismos
datos, mismos hiperparametros. Se construye **sobre el baseline ya arreglado
(3A)** -- `fix_entity_markers()` sigue aplicado, es la base mas correcta
disponible ahora mismo (ver `HALLAZGOS-BUGS-TOKENIZACION.md`).

**Por que class weights:** ataca el mismo problema que neg_ratio y focal loss
(4B) desde otro angulo -- las clases minoritarias (p.ej. `ALTERNATIVE_NAME`,
`APPLIED_TO`) tienen muy pocos ejemplos frente a `no_relation` o
`SUBCLASS_OF`, y con `CrossEntropyLoss` normal cada instancia pesa igual en
el gradiente. A diferencia de focal loss (que reescala segun que tan dificil
es CADA ejemplo, de forma dinamica durante el entreno), class weights
reescala segun la frecuencia de la CLASE, de forma estatica y fija desde el
principio -- mecanismo distinto, mismo objetivo.

La comparacion se hace contra **3A** (PubMedBERT + bugs arreglados, sin
typed markers, neg_ratio=3): argmax=0.3338, calibrado=0.4298 @ threshold=0.996.


## 1. Setup

In [ ]:
# Ejecucion en servidor local (zape), entorno conda "tfg". Mismo patron que 1G/2A/2B/3A/4A/4B.
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])


In [ ]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np
import torch.nn as nn

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_entity_markers
add_macro_f1_metric()
fix_entity_markers()   # <-- los dos fixes de HALLAZGOS-BUGS-TOKENIZACION.md, igual que 3A/4A/4B
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")


## 2. Configuracion

In [ ]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_classweights"
TECHNIQUE       = "CrossEntropyLoss(weight=inverse_freq) en vez de CrossEntropyLoss() -- sobre fix_entity_markers(), unico cambio vs 3A"

MAX_LENGTH     = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0

DATA_DIR    = Path("../data/english")
TRAIN_DATA  = DATA_DIR / "eng_train.txt"   # neg_ratio=3, sin cambios -- solo cambia la loss
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/4C-pubmedbert-classweights/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)


## 3. Calcular class weights

Formula "balanced" de `sklearn.utils.class_weight`: `weight[i] = n_total /
(n_clases * count[i])`. Se calcula sobre el train real (neg_ratio=3, sin
typed markers) -- las clases con pocos ejemplos reciben un peso mayor, las
clases con muchos ejemplos (dominadas por `no_relation`) reciben un peso
menor. El vector se pasa a `nn.CrossEntropyLoss(weight=...)`.

In [ ]:
train_instances = [json.loads(l) for l in open(TRAIN_DATA, encoding="utf-8") if l.strip()]
rel_counts = Counter(inst["relation"] for inst in train_instances)
n_total = len(train_instances)
n_classes = len(rel2id)

class_weights = np.zeros(n_classes, dtype=np.float32)
for rel, idx in rel2id.items():
    count = rel_counts.get(rel, 0)
    class_weights[idx] = n_total / (n_classes * count) if count > 0 else 0.0

print(f"Train: {n_total} instancias, {n_classes} clases\n")
print(f"{'Relacion':<22}{'count':>8}{'weight':>10}")
for rel, idx in sorted(rel2id.items(), key=lambda kv: rel_counts.get(kv[0], 0)):
    print(f"{rel:<22}{rel_counts.get(rel, 0):>8}{class_weights[idx]:>10.4f}")

CLASS_WEIGHTS = torch.tensor(class_weights, dtype=torch.float32)


## 4. Entrenamiento -- misma funcion que 1G/2A/2B/3A/4A/4B (`train_with_history`)

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history


In [ ]:
set_seed(SEED)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

framework.criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(device))   # <-- unico cambio real vs 3A
print(f"criterion reemplazado: {framework.criterion}")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev)={macro_f1_curado:.4f}")


## 5. Inferencia en blind + evaluacion oficial (argmax)

In [ ]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_raw), 64):
        batch = blind_raw[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")


## 6. Calibracion de threshold (grid fino) -- comparacion justa contra 3A

In [ ]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def f1_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)["macro_f1"]

FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, f1_at(all_probs, th)) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

# --- referencia: 3A (PubMedBERT + bugs arreglados, neg_ratio=3, sin typed markers) ---
BASELINE_ARGMAX = 0.333788238025547        # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_CALIBRADO = 0.42979245116674064   # outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json
BASELINE_TH = 0.996

print(f"\n{'':<28}{'argmax':>10}{'calibrado':>12}{'threshold':>12}")
print(f"{'3A (bugs arreglados, CE loss)':<28}{BASELINE_ARGMAX:>10.4f}{BASELINE_CALIBRADO:>12.4f}{BASELINE_TH:>12.3f}")
print(f"{'+ class weights':<28}{macro_f1_ciego_argmax:>10.4f}{macro_f1_ciego_calibrado:>12.4f}{best_threshold:>12.3f}")
print(f"{'delta':<28}{macro_f1_ciego_argmax-BASELINE_ARGMAX:>+10.4f}{macro_f1_ciego_calibrado-BASELINE_CALIBRADO:>+12.4f}")


## 7. Guardar resultados

In [ ]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": 3, "seed": SEED,
    },
    "class_weights": {rel: float(class_weights[idx]) for rel, idx in rel2id.items()},
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "baseline_comparison": {
        "source": "outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json (bugs arreglados, CrossEntropyLoss sin pesos)",
        "baseline_macro_f1_ciego_argmax": BASELINE_ARGMAX,
        "baseline_macro_f1_ciego_calibrado": BASELINE_CALIBRADO,
        "delta_argmax": macro_f1_ciego_argmax - BASELINE_ARGMAX,
        "delta_calibrado": macro_f1_ciego_calibrado - BASELINE_CALIBRADO,
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
print(json.dumps(results, indent=2, ensure_ascii=False))


## 8. Conclusion

*(rellenar tras ejecutar con el resultado real -- no se pone ningun numero aqui
de antemano, igual que 2A/2B)*